# 03 · Approach 1: TextCNN, local n-gram patterns over the word sequence
Owner: **Member 3** · Runtime: Colab **GPU** (T4)

**Why this model.** News categories are signalled by short phrases: *ligi kuu* (premier league),
*benki kuu* (central bank), *waziri wa fedha* (finance minister). A 1-D convolution slides filters of
width 3, 4 and 5 words along the article and fires on such phrases wherever they appear; max-pooling
over time keeps the strongest match (Kim, 2014). It models **local** word order, but not dependencies
between distant parts of the article.

Experiments change one thing at a time; each is motivated by an EDA finding or a previous run.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv` and `SampleSubmission.csv` (from Zindi) in a
Google Drive folder called `swahili_news`, and set `REPO_URL` below.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-news-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-news-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_news")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    for name in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        shutil.copy(DRIVE_DATA / name, REPO_DIR / "data" / name)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")             # e.g. downloaded word vectors persist
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")   # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

from src import config as C
from src.data import get_splits, split_frames, encode_labels, load_unlabelled
from src.datasets import TokenDataset, pad_collate
from src.models import TextCNN
from src.pipeline import run_experiments, finalize
from src.text import tokenize, Vocab, encode_corpus, load_fasttext_vectors
from src.utils import set_seed, get_device

MEMBER, MODEL = "Member 3", "TextCNN"
set_seed(C.SEED)
print("Device:", get_device())
df = get_splits()
tr, va, te = split_frames(df)
ytr, yva, yte = (encode_labels(d.label) for d in (tr, va, te))
zindi = load_unlabelled()
K = len(C.CLASSES)
vocab = Vocab(tr.content.map(tokenize))              # built on TRAIN only
print("vocabulary:", len(vocab), "| validation OOV rate:", round(vocab.oov_rate(va.content.map(tokenize)), 4))

## Sequence representation: word ids, truncated to `MAX_WORDS`

In [ ]:
_cache = {}


def seqs(max_len):
    if max_len not in _cache:
        _cache[max_len] = {n: encode_corpus(d.content, vocab, max_len) for n, d in
                           {"tr": tr, "va": va, "te": te, "z": zindi}.items()}
    return _cache[max_len]

## Optional: pretrained Swahili fastText vectors (Grave et al., 2018)
Trained on Common Crawl + Wikipedia with sub-word information, so they cover inflected forms the
training set rarely shows. The download is a few hundred MB and is cached on Drive. If it fails, the
pretrained experiment is skipped.

In [ ]:
VEC_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.sw.300.vec.gz"
vec_path = Path(C.CACHE_DIR) / "cc.sw.300.vec.gz"
EMB = None
try:
    if not vec_path.exists() and not C.QUICK:
        print("Downloading", VEC_URL)
        urllib.request.urlretrieve(VEC_URL, vec_path)
    if vec_path.exists():
        EMB, cov = load_fasttext_vectors(vocab, vec_path, 300)
        print(f"Pretrained vectors found for {cov:.1%} of the vocabulary")
except Exception as ex:
    print("Pretrained vectors unavailable:", ex)

In [ ]:
def make_datasets(p):
    s = seqs(p.get("max_len", C.MAX_WORDS))
    return TokenDataset(s["tr"], ytr, p.get("word_dropout", 0.0)), TokenDataset(s["va"], yva)


def build_model(p):
    pre = EMB if p.get("pretrained") else None
    return TextCNN(len(vocab), K, dim=300 if pre is not None else p.get("dim", 128),
                   kernels=p.get("kernels", (3, 4, 5)), filters=p.get("filters", 128),
                   dropout=p.get("dropout", 0.5), pretrained=pre)

## Experiments

In [ ]:
DEFAULT_TRAIN = dict(epochs=20, lr=2e-3, batch_size=32, weight_decay=1e-4, patience=4, num_workers=0)
EXPERIMENTS = [
    dict(id="CNN-01", description="TextCNN, random embeddings, kernels 3/4/5, 512 words",
         hypothesis="Local phrase detectors are enough to recognise most categories.",
         motivated_by="Starting point (compare with BL-02)", params={}),
    dict(id="CNN-02", description="+ class-weighted loss",
         hypothesis="Weighting rare classes raises their recall (macro-F1) at some cost in log loss.",
         motivated_by="EDA 2 (118:1 imbalance) + CNN-01 per-class recall",
         params={}, train={"class_weights": True}),
    dict(id="CNN-03", description="+ word dropout 0.1 (augmentation)",
         hypothesis="Randomly hiding words stops the CNN from relying on a few names (e.g. 'simba', 'yanga').",
         motivated_by="CNN-01 learning curves: fast overfitting on a small training set",
         params={"word_dropout": 0.1}),
    dict(id="CNN-04", description="Only the first 256 words",
         hypothesis="Shorter input loses little because key information is early; training is faster.",
         motivated_by="EDA 4/8 + BL-04 vs BL-05 (head vs tail)", params={"max_len": 256}),
    dict(id="CNN-05", description="Wider receptive field: kernels 2/3/4/5/7",
         hypothesis="Longer phrases (titles, organisation names) need wider filters.",
         motivated_by="Architecture check on phrase length", params={"kernels": (2, 3, 4, 5, 7)}),
]
if EMB is not None:
    EXPERIMENTS.append(dict(id="CNN-06", description="Pretrained fastText Swahili embeddings (fine-tuned)",
                            hypothesis="Vectors learned on a large Swahili corpus help rare and unseen inflected forms.",
                            motivated_by="EDA 5: 53% hapax types, ~4.7% validation OOV",
                            params={"pretrained": True, "word_dropout": 0.1}))
best, summary = run_experiments(EXPERIMENTS, build_model, make_datasets, member=MEMBER, model_name=MODEL,
                                n_classes=K, y_train=ytr, default_train=DEFAULT_TRAIN, collate_fn=pad_collate)
summary.round(4)

> Fill the `observation` column in `experiments/log_textcnn.csv`. If a result surprises you, add a
> follow-up experiment (e.g. combine the two best changes) and re-run: that is the progression the
> rubric asks for.

## Final evaluation on the test split (once)

In [ ]:
s = seqs(best["exp"]["params"].get("max_len", C.MAX_WORDS))
model, proba_test, metrics = finalize(best, build_model, TokenDataset(s["te"], yte), yte, te.id.values, C.CLASSES,
                                      key="textcnn", display_name="TextCNN", collate_fn=pad_collate,
                                      zindi_ds=TokenDataset(s["z"]), zindi_df=zindi)

In [ ]:
from src.utils import persist_outputs
persist_outputs()